### RAG Pipelines - Data Ingestion to Vector DB Pipeline


In [1]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/h6/fs4ps3sd2dv39whm40sg9ccm0000gn/T/ipykernel_2834/2016311095.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, TextLoader
/Users/pranavisai/Documents/RAG/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read the pdfs inside the data/pdf directory

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in the directory.")

    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file.name}")
        try:
            # Load the PDF using PyMuPDFLoader
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            all_documents.extend(documents)
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
            try:
                loader = PyPDFLoader(str(pdf_file))
                documents = loader.load()
                for doc in documents:
                    doc.metadata["pdf_source"] = pdf_file.name
                    doc.metadata["file_type"] = 'pdf'
                all_documents.extend(documents)
                print(f"Successfully processed {pdf_file.name} with PyPDFLoader.")
            except Exception as e:
                print(f"Error processing {pdf_file.name} with PyPDFLoader: {e}")

    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

pdf_directory = "../data/pdf"
documents = process_all_pdfs(pdf_directory)

Found 4 PDF files in the directory.
Processing file: arc_Cover_letter.pdf
Processing file: ailoys_Cover_letter.pdf
Processing file: data_engineer_thryve_Cover_letter.pdf
Processing file: SE_thryve_Cover_letter.pdf
Total documents loaded: 4


In [3]:
documents

[Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '../data/pdf/arc_Cover_letter.pdf', 'file_path': '../data/pdf/arc_Cover_letter.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'arc_Cover_letter', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Dear Clemens Wessendorff, \nI hope you are doing well. \nI am writing to apply for the Fullstack Engineer position. My cover letter is split into three parts, a \nbrief overview of why I am looking for a new position, what my strengths are, and how I can fit in \nat ARC Intelligence GmbH. \nMost recently, I worked at Morpheus Space, where I co-led the development and stabilization of \nproduction software in the aerospace sector. Prior to that, I developed full-stack web \napplications, automation platforms, ETL pipelines, and backend services using Python, \nJavaScript, SQL, and Linux. Du

In [4]:
### Text splitting to get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""])
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into total chunks created: {len(split_docs)}")

    if split_docs:
        print(f"\nExample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print the first 500 characters of the first chunk
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [5]:
chunks = split_documents(documents)
chunks

Split 4 documents into total chunks created: 9

Example Chunk:
Content: Dear Clemens Wessendorff, 
I hope you are doing well. 
I am writing to apply for the Fullstack Engineer position. My cover letter is split into three parts, a 
brief overview of why I am looking for a...
Metadata: {'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '../data/pdf/arc_Cover_letter.pdf', 'file_path': '../data/pdf/arc_Cover_letter.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'arc_Cover_letter', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}


[Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '../data/pdf/arc_Cover_letter.pdf', 'file_path': '../data/pdf/arc_Cover_letter.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'arc_Cover_letter', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Dear Clemens Wessendorff, \nI hope you are doing well. \nI am writing to apply for the Fullstack Engineer position. My cover letter is split into three parts, a \nbrief overview of why I am looking for a new position, what my strengths are, and how I can fit in \nat ARC Intelligence GmbH. \nMost recently, I worked at Morpheus Space, where I co-led the development and stabilization of \nproduction software in the aerospace sector. Prior to that, I developed full-stack web \napplications, automation platforms, ETL pipelines, and backend services using Python, \nJavaScript, SQL, and Linux. Du

In [6]:
### embedding and vector store creation

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'): #this model is available in hugging-face and is responsible for converting text into vector embeddings. It has around 384 dimensions and is designed for efficient semantic search and clustering tasks.
        self.model_name = model_name
        self.model = None
        self._load_model() # _ indicates that this method is intended for internal use within the class and is not meant to be called from outside the class. It is a convention in Python to indicate that a method or attribute is private or internal to the class. It is a protected function.

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Successfully loaded embedding model: {self.model.get_sentence_embedding_dimension()} dimensions")
        except Exception as e:
            print(f"Error loading embedding model: {e}")
            raise 

    def generate_embeddings(self, texts: List[str]) -> np.ndarray: #generates a list of text and returns a numpy array of embeddings. Each row in the returned array corresponds to the embedding of the respective text in the input list.
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Successfully generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise

## initialize the embedding manager

embedding_manager = EmbeddingManager(model_name='all-MiniLM-L6-v2')
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6806.79it/s]


Successfully loaded embedding model: 384 dimensions


/var/folders/h6/fs4ps3sd2dv39whm40sg9ccm0000gn/T/ipykernel_2834/645911786.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Successfully loaded embedding model: {self.model.get_sentence_embedding_dimension()} dimensions")


### Vector Store

In [8]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_chunks", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF chunks for RAG system"})
            print(f"Initialized vector store with collection: {self.collection_name}")
            print(f"Exisiting documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            ## preparing metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list for JSON serialization

            try:
                self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text    
                )
                print(f"Successfully added {len(documents)} documents to vector store.")
                print(f"Total documents in collection after addition: {self.collection.count()}")

            except Exception as e:
                print(f"Error adding documents to vector store: {e}")
                raise
vectorstore = VectorStore()
vectorstore


Initialized vector store with collection: pdf_chunks
Exisiting documents in collection: 9


In [9]:
chunks

[Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '../data/pdf/arc_Cover_letter.pdf', 'file_path': '../data/pdf/arc_Cover_letter.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'arc_Cover_letter', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Dear Clemens Wessendorff, \nI hope you are doing well. \nI am writing to apply for the Fullstack Engineer position. My cover letter is split into three parts, a \nbrief overview of why I am looking for a new position, what my strengths are, and how I can fit in \nat ARC Intelligence GmbH. \nMost recently, I worked at Morpheus Space, where I co-led the development and stabilization of \nproduction software in the aerospace sector. Prior to that, I developed full-stack web \napplications, automation platforms, ETL pipelines, and backend services using Python, \nJavaScript, SQL, and Linux. Du

In [10]:
### convert the text to embeddings
texts = [chunk.page_content for chunk in chunks]
texts

['Dear Clemens Wessendorff, \nI hope you are doing well. \nI am writing to apply for the Fullstack Engineer position. My cover letter is split into three parts, a \nbrief overview of why I am looking for a new position, what my strengths are, and how I can fit in \nat ARC Intelligence GmbH. \nMost recently, I worked at Morpheus Space, where I co-led the development and stabilization of \nproduction software in the aerospace sector. Prior to that, I developed full-stack web \napplications, automation platforms, ETL pipelines, and backend services using Python, \nJavaScript, SQL, and Linux. Due to company-wide budget reductions, my position at Morpheus \nSpace was recently eliminated.I am now looking for an opportunity where engineers are \nencouraged to take ownership, and continuously improve the product. \nBelow is a summary of what I bring and how it aligns with your requirements: \nMy Strengths \nHow this helps the team \nFull-Stack Development',
 'Below is a summary of what I bring

In [11]:
### generate the embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)
embeddings

### store the chunks and their corresponding embeddings in the vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 9 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.30it/s]

Successfully generated embeddings with shape: (9, 384)
Adding 9 documents to vector store...
Successfully added 9 documents to vector store.
Total documents in collection after addition: 10
Successfully added 9 documents to vector store.
Total documents in collection after addition: 11
Successfully added 9 documents to vector store.
Total documents in collection after addition: 12
Successfully added 9 documents to vector store.
Total documents in collection after addition: 13
Successfully added 9 documents to vector store.
Total documents in collection after addition: 14
Successfully added 9 documents to vector store.
Total documents in collection after addition: 15
Successfully added 9 documents to vector store.
Total documents in collection after addition: 16
Successfully added 9 documents to vector store.
Total documents in collection after addition: 17
Successfully added 9 documents to vector store.
Total documents in collection after addition: 18


### RAG Retriever from VectorStore

In [22]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager


    ## This function is used to retrieve relevant chunks from the vector store based on a user query. It takes a query string and an 
    # optional parameter top_k which specifies how many of the most relevant chunks to return. 
    # The function generates an embedding for the query, retrieves all document embeddings from the vector store, 
    # computes cosine similarity between the query embedding and document embeddings, and returns the top_k most 
    # similar documents along with their metadata and similarity scores.
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving relevant chunks for query: '{query}'")
        try:
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]  # Get the embedding for the query
            
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
                )
            
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                scores = results['distances'][0]
                ids = results['ids'][0]
                for i, (doc_id, document, metadata, score) in enumerate(zip(ids, documents, metadatas, scores)):
                    similarity_score = 1 - score
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "score": score,
                            "rank": i + 1
                        })
                        print(f"Retrieved document ID: {doc_id} with similarity score: {similarity_score:.4f}")
                    else:
                        print(f"Not found")

                    return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise

rag_retriever = RAGRetriever(vector_store=vectorstore, embedding_manager=embedding_manager)

In [23]:
rag_retriever

In [26]:
### RAG pipeline with groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### initialize the groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in environment variables. Please set it in your .env file.")

llm = ChatGroq(api_key=groq_api_key, model="gemma2-9b-it", temperature=0.1, max_tokens=1024)

## RAG function created
def rag_function(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context= "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant information found in the documents."
    ## generate the answer using groq LLM in case context not found
    prompt = f"Use the following context to answer the question:\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content